In [15]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [16]:
import kagglehub
path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ibm-hr-analytics-attrition-dataset' dataset.
Path to dataset files: /kaggle/input/ibm-hr-analytics-attrition-dataset


In [17]:
cd /kaggle/input/ibm-hr-analytics-attrition-dataset

/kaggle/input/ibm-hr-analytics-attrition-dataset


In [18]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

## Preprocessing

In [19]:
# 1. target handling

df['target'] = df['Attrition'].astype(bool).astype(int)
df = df.drop(columns=['Attrition'])

In [20]:
# Numerical columns (int64)
num_cols = [
    'Age', 'DailyRate', 'DistanceFromHome', 'Education',
    'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate',
    'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
    'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating',
    'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
    'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
    'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
    'YearsWithCurrManager', 'target'
]

# Non-numerical columns (object/categorical)
cat_cols = [
    'BusinessTravel', 'Department', 'EducationField', 'Gender',
    'JobRole', 'MaritalStatus', 'Over18', 'OverTime'
]

In [21]:
df[num_cols].head(3)

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,target
0,41,1102,1,2,1,1,2,94,3,2,4,5993,19479,8,11,3,1,80,0,8,0,1,6,4,0,5,1
1,49,279,8,1,1,2,3,61,2,2,2,5130,24907,1,23,4,4,80,1,10,3,3,10,7,1,7,1
2,37,1373,2,2,1,4,4,92,2,1,3,2090,2396,6,15,3,2,80,0,7,3,3,0,0,0,0,1


In [22]:
# 2. Check for missing values

df.isnull().sum().sum()

np.int64(0)

In [23]:
# 3. Drop constant/low-variance columns using VarianceThreshold
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.05) 
vt.fit(df[num_cols])

dropped_num = [c for c, s in zip(num_cols, vt.get_support()) if not s]
dropped_cat = [c for c in cat_cols if df[c].nunique() <= 1]

id_cols = ['EmployeeNumber']

drop_cols = dropped_num + dropped_cat + id_cols
df = df.drop(columns=drop_cols)

num_cols = [c for c in num_cols if c not in drop_cols]
cat_cols = [c for c in cat_cols if c not in drop_cols]

print(f"Dropped (zero-variance numerical): {dropped_num}")
print(f"Dropped (constant categorical):    {dropped_cat}")
print(f"Dropped (ID columns):              {id_cols}")
print(f"\nRemaining columns: {len(df.columns)}")
print(f"Numerical: {len(num_cols)}, Categorical: {len(cat_cols)}")

Dropped (zero-variance numerical): ['EmployeeCount', 'StandardHours', 'target']
Dropped (constant categorical):    ['Over18']
Dropped (ID columns):              ['EmployeeNumber']

Remaining columns: 30
Numerical: 23, Categorical: 7


In [24]:
# 4. Encode categorical variables (one-hot encoding)

df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

print(f"Shape after encoding: {df.shape}")
df.head(3)

Shape after encoding: (1470, 44)


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Male,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,41,1102,1,2,2,94,3,2,4,5993,19479,8,11,3,1,0,8,0,1,6,4,0,5,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1
1,49,279,8,1,3,61,2,2,2,5130,24907,1,23,4,4,1,10,3,3,10,7,1,7,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0
2,37,1373,2,2,4,92,2,1,3,2090,2396,6,15,3,2,0,7,3,3,0,0,0,0,0,1,1,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,1,1


In [25]:
# 5. Split features and target

X = df.drop(columns=['target'])
y = df['target']

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts(normalize=True)}")

KeyError: "['target'] not found in axis"

In [ ]:
# 6. Train/test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# 7. Feature scaling (fit on train, transform both)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_scaled.head(3)